<a href="https://colab.research.google.com/github/danielchin-ck/ADALL_github/blob/main/Project/6096089W_CDA1C03_ADALL_Project_2026-Ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Libraries used in this notebook
import os
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Modelling libraries
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Make wide tables easier to read in Colab
pd.set_option('display.max_columns', 100)

In [2]:
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
assert token, 'Missing Colab Secret: GITHUB_TOKEN'

os.environ['GITHUB_TOKEN'] = token
os.environ['GITHUB_USER'] = 'danielchin-ck'
os.environ['GITHUB_REPO'] = 'ADALL_github'
os.environ['GITHUB_EMAIL'] = 'danielchin.ck@gmail.com'

print('GitHub settings loaded. Token is not printed.')

In [3]:
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

if RUN_API_CELLS == True:
    from google.colab import userdata
    from openai import OpenAI

    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print('OpenAI client is ready.')
else:
    print('Manual chatbot mode. Copy the prompts when they appear.')



OpenAI client is ready.


In [13]:
#Version 1
bp_prompt = f"""
You are a Data Scientist of specialized experience in tree-based regression models (XGBoost, LightGBM, Random Forest) for actuarial and real-estate forecasting.
Your expertise is translating ambiguous socio-economic problems into rigorous ML objectives with clear business impact.

CONTEXT:
Singaporean seniors face a critical retirement challenge: their HDB flat—their largest asset—is depreciating as the lease approaches the 40-year mark.
Their monthly income is depleting in retirement. One solution is to downsize their existing HDB flat to unlock cash value.

YOUR TASK:
Craft a complete ML modelling brief that addresses the following:

1. BUSINESS PROBLEM DEFINITION
   •	Describe the business problem you aim to address. Explain why it matters and who is affected.

2. PRIMARY BENEFICIARY PERSONA
   •	Identify the key beneficiary of the ML solution. Create a simple persona that captures details important for this project,
   such as the person’s goals, pain points, behaviours and how they would use the solution.

3. JTBD-FRAMED MODELLING OBJECTIVES
   •	Use the Job-To-Be-Done (JTBD) framework to shape your modelling objectives. State the job the user is trying to complete,
   and define your target variable and likely predictors based on this job.

FORMAT:
Respond in point forms

CONSTRAINTS:
Keep it concise and to the point.
Limit each section to maximum 200 characters per line.
"""

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=bp_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')



LLM response:
- **1) BUSINESS PROBLEM DEFINITION**
  - Seniors nearing 40-yr HDB lease face asset value depreciation.
  - Income runs down; cash needs rise.
  - They downsize to unlock equity, but decision timing is risky.
  - **Goal:** predict downsizing action + expected cash impact to guide timing.
  - **Who affected:** seniors, families, HDB transfer/downsizing service providers, policymakers.

- **2) PRIMARY BENEFICIARY PERSONA**
  - **Persona:** “Retiring Lee” (68, HDB lease ~30–38 yrs, fixed monthly income)
  - **Goal:** fund daily living + medical costs without selling at worst timing.
  - **Pain points:** uncertainty on lease value, cash shortfall, stress choosing right move.
  - **Behaviour:** seeks estimates online/at agencies; acts when “feels safe”.
  - **Use:** enters current flat details → gets lease-sensitive downsizing guidance + cash estimate.

- **3) JTBD-FRAMED MODELLING OBJECTIVES**
  - **JTBD:** “Estimate whether/when downsizing my HDB to access cash, given my le

In [4]:
#Version 2
bp_prompt = f"""
You are a Data Scientist of specialized experience in tree-based regression models (XGBoost, LightGBM, Random Forest) for actuarial and real-estate forecasting.
Your expertise is translating ambiguous socio-economic problems into rigorous ML objectives with clear business impact.

CONTEXT:
Singaporean seniors face a critical retirement challenge: their HDB flat—their largest asset—is depreciating as the lease approaches the 40-year mark.
Their monthly income is depleting in retirement. One solution is to downsize their existing HDB flat to unlock cash value.

YOUR TASK:
Craft a complete ML modelling brief that addresses the following:

1. BUSINESS PROBLEM DEFINITION
   •	Describe the business problem you aim to address. Explain why it matters and who is affected.

2. PRIMARY BENEFICIARY PERSONA
   •	Identify the key beneficiary of the ML solution. Create a simple persona that captures details important for this project,
   such as the person’s goals, pain points, behaviours and how they would use the solution.

3. JTBD-FRAMED MODELLING OBJECTIVES
   •	Use the Job-To-Be-Done (JTBD) framework to shape your modelling objectives. State the job the user is trying to complete,
   and define your target variable and likely predictors based on this job.

FORMAT:
Respond in point forms

CONSTRAINTS:
Keep it concise and to the point.
Limit each section to maximum 200 characters per line.
"""

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=bp_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')



LLM response:
- **1) BUSINESS PROBLEM DEFINITION**
  - Seniors’ HDB lease decay reduces asset value near/after 40 years.
  - Downsizing can release cash to fund retirement when income depletes.
  - We build a model to recommend who should downsize now, and estimate expected value/cash uplift.
  - Affected: seniors, family caregivers, HDB/financial advisors, retirement planners.

- **2) PRIMARY BENEFICIARY PERSONA**
  - **Persona: “Retiring Homeowner”**
  - Age 60–75, owns HDB (≈20–35 yrs lease left).
  - Goal: maximize retirement cash runway while minimizing disruption/risk.
  - Pain: uncertain resale value, lease decline fear, affordability of next home.
  - Behaviour: compares options via brokers, spreadsheets, advice; delays until “urgent”.
  - Uses ML: gets “downsize now vs later” guidance + estimated proceeds & fit.

- **3) JTBD-FRAMED MODELLING OBJECTIVES**
  - **JTBD:** “Decide the best time to downsize my HDB to fund retirement.”
  - **Primary target (choose one):**
    - **T1